# V1 Legal NER -- decoder repair cells (inference-only)

**Insert after the cell that defines `oof_window_records_by_doc` and `ensemble_test_window_records_by_doc` (cell 99 of the five-fold notebook), before final package selection.**
The `insert_decoder_repair_cells.py` script does this for you; or run these four cells in the live kernel (or `%run -i` the combined `.py`).

| cell | what it does | needs kernel state? |
|---|---|---|
| library | BIO-constrained Viterbi, word/punctuation units, boundary policies, strict scorer, nested-CV helpers, ZIP checker | no |
| synthetic tests | 19 tests on **invented** examples; raises if any fails | no |
| kernel evaluation | re-scores the baseline (must reproduce the supplied OOF metrics), evaluates 7 decoder structures + per-label bias tuning with **nested fold-wise validation**, applies a predeclared gate | yes (cached OOF probabilities) |

No model training, no new weights, V1 checkpoint and `bpe_en_25k.json` untouched. A ZIP is written **only** if the nested gate passes; otherwise only diagnostics are saved. A passing OOF gate is not evidence of a public-leaderboard gain.

## 1. Library (pure functions)

In [ ]:
# =====================================================================
# V1 Legal NER -- decoder repair LIBRARY (pure functions, no notebook state)
# Inference-only. No model training. No pretrained weights. Safe to run anywhere.
# =====================================================================
import re, bisect, json, os, zipfile, unicodedata
import numpy as np

# ---------------------------------------------------------------- scheme
class BioScheme:
    """Order-agnostic description of the BIO tag set (works with the notebook's BIO_LABELS)."""

    def __init__(self, bio_labels):
        self.tags = list(bio_labels)
        self.id_of = {t: i for i, t in enumerate(self.tags)}
        if "O" not in self.id_of:
            raise ValueError("BIO tag set has no 'O'.")
        self.labels = [t[2:] for t in self.tags if t.startswith("B-")]
        for lab in self.labels:
            if f"I-{lab}" not in self.id_of:
                raise ValueError(f"Missing I-{lab}.")
        if len(self.tags) != 1 + 2 * len(self.labels):
            raise ValueError("Unexpected tags besides O/B-X/I-X.")
        self.k = len(self.tags)
        self.o_id = self.id_of["O"]
        self.b_ids = np.array([self.id_of[f"B-{l}"] for l in self.labels], dtype=np.int64)
        self.i_ids = np.array([self.id_of[f"I-{l}"] for l in self.labels], dtype=np.int64)


# ---------------------------------------------------------------- Viterbi
def constrained_viterbi(scores, scheme):
    """Max-score tag path with BIO validity: a path may start with O or B-X only,
    and I-X may only follow B-X or I-X. O(T*K), numpy-vectorised over classes."""
    s = np.asarray(scores, dtype=np.float64)
    if s.ndim != 2 or s.shape[1] != scheme.k:
        raise ValueError(f"Expected [T,{scheme.k}] scores, got {s.shape}.")
    n = s.shape[0]
    if n == 0:
        return []
    NEG = -1e18
    o, B, I = scheme.o_id, scheme.b_ids, scheme.i_ids
    dp = np.full(scheme.k, NEG)
    dp[o] = s[0, o]
    dp[B] = s[0, B]
    back = np.zeros((n, scheme.k), dtype=np.int32)
    for t in range(1, n):
        bp = int(dp.argmax())
        bv = dp[bp]
        new = np.empty(scheme.k)
        new[o] = bv + s[t, o]
        back[t, o] = bp
        new[B] = bv + s[t, B]
        back[t, B] = bp
        pb, pi = dp[B], dp[I]
        use_b = pb >= pi
        new[I] = np.where(use_b, pb, pi) + s[t, I]
        back[t, I] = np.where(use_b, B, I)
        dp = new
    path = [int(dp.argmax())]
    for t in range(n - 1, 0, -1):
        path.append(int(back[t, path[-1]]))
    return path[::-1]


def bias_vector(scheme, bias_by_label=None, o_bias=0.0):
    """Additive score bias: beta_X is added to both B-X and I-X (positive = more X)."""
    v = np.zeros(scheme.k, dtype=np.float64)
    v[scheme.o_id] = float(o_bias)
    for lab, beta in (bias_by_label or {}).items():
        v[scheme.id_of[f"B-{lab}"]] += float(beta)
        v[scheme.id_of[f"I-{lab}"]] += float(beta)
    return v


# ---------------------------------------------------------------- inputs
def validate_probs(p, k, expected_len=None, atol=2e-3):
    a = np.asarray(p, dtype=np.float64)
    if a.size == 0 and a.ndim < 2:
        a = a.reshape(0, k)          # empty doc / zero-token window
    if a.ndim != 2 or a.shape[1] != k:
        raise ValueError(f"Expected [tokens,{k}] probabilities, got {a.shape}.")
    if expected_len is not None and a.shape[0] != expected_len:
        raise ValueError("Probability/offset length mismatch.")
    if a.shape[0]:
        if not np.isfinite(a).all():
            raise ValueError("Non-finite probabilities.")
        if (a < -1e-6).any() or (a > 1.0001).any():
            raise ValueError("Scores do not look like probabilities (logits?).")
        if not np.allclose(a.sum(axis=1), 1.0, atol=atol):
            raise ValueError("Rows do not sum to 1 (expected softmax probabilities).")
    return np.clip(a, 1e-8, 1.0)


def clean_offset(off, text_len):
    """Return (start,end) clipped to the text, or None for special/zero-width/invalid offsets."""
    if off is None or not isinstance(off, (list, tuple)) or len(off) < 2:
        return None
    a, b = off[0], off[1]
    if a is None or b is None:
        return None
    a, b = int(a), int(b)
    if a < 0 or a >= b or a >= text_len:
        return None
    return (a, min(b, text_len))


# ---------------------------------------------------------------- units
def _wordish(ch):
    return ch.isalnum() or ch == "_" or unicodedata.category(ch).startswith("M")


def _is_join(ch):
    c = unicodedata.category(ch)
    return c.startswith("M") or c == "Cf"       # combining marks, virama, ZWJ/ZWNJ


def make_units(text, mode):
    """'whitespace': maximal non-space runs.  'word_punct': word runs, and each
    punctuation character on its own. Combining marks / joiners (e.g. Devanagari vowel signs,
    viramas, decomposed accents) are kept inside their word."""
    if mode == "whitespace":
        return [(m.start(), m.end()) for m in re.finditer(r"\S+", text)]
    if mode != "word_punct":
        raise ValueError(f"Unknown unit mode {mode}.")
    units = []
    for m in re.finditer(r"\w+|[^\w\s]", text):
        s, e = m.start(), m.end()
        if units and units[-1][1] == s:
            prev_last, cur_first = text[s - 1], text[s]
            attach_mark = e - s == 1 and _is_join(cur_first) and _wordish(prev_last)
            continue_after_mark = _is_join(prev_last) and _wordish(cur_first) and _wordish(text[units[-1][0]])
            if attach_mark or continue_after_mark:
                units[-1] = (units[-1][0], e)
                continue
        units.append((s, e))
    return units


def assign_tokens_to_units(offsets, units):
    """Each token goes to the unit it overlaps most. Returns (groups, n_straddling_tokens)."""
    groups = [[] for _ in units]
    straddle = 0
    if not units:
        return groups, 0
    starts = [s for s, _ in units]
    ends = [e for _, e in units]
    for ti, off in enumerate(offsets):
        if off is None:
            continue
        ts, te = off
        wi = bisect.bisect_right(ends, ts)
        best, bo, cnt = None, 0, 0
        while wi < len(units) and starts[wi] < te:
            ov = min(te, ends[wi]) - max(ts, starts[wi])
            if ov > 0:
                cnt += 1
                if ov > bo:
                    best, bo = wi, ov
            wi += 1
        if best is not None:
            groups[best].append(ti)
        if cnt > 1:
            straddle += 1
    return groups, straddle


def unit_score_matrix(logp, groups, scheme, empty_entity_score=-1e9):
    """Per-unit class scores from BPE-piece log-probs.
    O: mean log P(O).  I-X: mean log P(I-X) over the unit's pieces.
    B-X: first piece B-X, remaining pieces I-X (the way a multi-piece word-start is tagged)."""
    U = len(groups)
    S = np.full((U, scheme.k), empty_entity_score, dtype=np.float64)
    S[:, scheme.o_id] = 0.0
    B, I = scheme.b_ids, scheme.i_ids
    for ui, idx in enumerate(groups):
        if not idx:
            continue
        lp = logp[idx]
        n = len(idx)
        S[ui, scheme.o_id] = lp[:, scheme.o_id].mean()
        S[ui, I] = lp[:, I].mean(axis=0)
        S[ui, B] = (lp[0, B] + lp[1:, I].sum(axis=0)) / n
    return S


# ---------------------------------------------------------------- prepare (cacheable)
def prepare_document(probs, offsets, text, scheme, mode):
    """Aggregate pieces -> scoring runs. Cached once per (doc, mode); bias/policy applied later."""
    offsets = list(offsets)
    p = validate_probs(probs, scheme.k, expected_len=len(offsets))
    cleaned = [clean_offset(o, len(text)) for o in offsets]
    diag = {"tokens": len(offsets), "null_offsets": sum(c is None for c in cleaned),
            "units": 0, "empty_units": 0, "straddling_tokens": 0, "unassigned_tokens": 0}
    valid = [i for i, c in enumerate(cleaned) if c is not None]
    # keep positional order even if an aggregator hands back unsorted pieces
    valid.sort(key=lambda i: cleaned[i])
    logp = np.log(p)
    if mode == "token":
        runs, cur = [], []
        for i in valid:
            if cur:
                prev_end = cleaned[cur[-1]][1]
                gap = text[prev_end:cleaned[i][0]] if cleaned[i][0] > prev_end else ""
                if gap.strip():           # uncovered non-space text => no evidence => break the run
                    runs.append(cur)
                    cur = []
            cur.append(i)
        if cur:
            runs.append(cur)
        return {"mode": mode, "diag": diag,
                "runs": [(logp[idx], [cleaned[i] for i in idx]) for idx in runs]}
    units = make_units(text, mode)
    sub_offs = [cleaned[i] for i in valid]
    groups_local, straddle = assign_tokens_to_units(sub_offs, units)
    groups = [[valid[j] for j in g] for g in groups_local]
    diag.update(units=len(units), empty_units=sum(1 for g in groups if not g),
                straddling_tokens=straddle,
                unassigned_tokens=len(valid) - sum(len(g) for g in groups))
    if not units or not valid:
        return {"mode": mode, "diag": diag, "runs": []}
    return {"mode": mode, "diag": diag, "runs": [(unit_score_matrix(logp, groups, scheme), units)]}


# ---------------------------------------------------------------- boundary policies
_SOFT = ",;:\"'“”‘’"
_DASH = "-–—"
_LEGACY = " \t\n\r.,;:()[]{}'\""
_OPEN, _CLOSE = "([{", ")]}"
_ABBREV = {"ltd", "ors", "anr", "co", "inc", "pvt", "no", "dr", "mr", "mrs", "ms", "shri", "smt",
           "vs", "v", "hon", "jr", "sr", "st", "govt", "dept", "corp", "assn", "addl", "asst"}


def _keep_trailing_period(seg):
    core = seg[:-1].rstrip()
    if not core:
        return False
    last = re.split(r"[\s,;(\[]+", core)[-1]
    return ("." in last) or (len(last) == 1 and last.isalpha()) or last.lower() in _ABBREV


def trim_span(text, start, end, policy):
    """Edge adjustment. Policies:
       preserve : whitespace only (the model's own boundaries stand)
       soft     : + commas/semicolons/colons/quotes/dashes (never periods or brackets)
       smart    : soft + unbalanced edge brackets + trailing '.' unless abbreviation-like
       legacy   : the original unconditional strip of  .,;:()[]{}'"  (kept for comparison only)"""
    start, end = max(0, int(start)), min(len(text), int(end))
    ws = lambda ch: ch.isspace()
    while start < end and ws(text[start]):
        start += 1
    while start < end and ws(text[end - 1]):
        end -= 1
    if policy == "preserve":
        return (start, end) if start < end else None
    if policy == "legacy":
        while start < end and text[start] in _LEGACY:
            start += 1
        while start < end and text[end - 1] in _LEGACY:
            end -= 1
        return (start, end) if start < end else None
    if policy not in ("soft", "smart"):
        raise ValueError(f"Unknown edge policy {policy}.")
    soft = _SOFT + _DASH
    changed = True
    while changed and start < end:
        changed = False
        while start < end and (ws(text[start]) or text[start] in soft):
            start += 1; changed = True
        while start < end and (ws(text[end - 1]) or text[end - 1] in soft):
            end -= 1; changed = True
        if policy == "soft" or start >= end:
            continue
        seg = text[start:end]
        first, last = seg[0], seg[-1]
        if last in _OPEN:                                     # trailing opener can never pair
            end -= 1; changed = True; continue
        if first in _CLOSE:                                   # leading closer can never pair
            start += 1; changed = True; continue
        if last in _CLOSE and seg.count(_OPEN[_CLOSE.index(last)]) < seg.count(last):
            end -= 1; changed = True; continue
        if first in _OPEN and seg.count(first) > seg.count(_CLOSE[_OPEN.index(first)]):
            start += 1; changed = True; continue
        if last == "." and not _keep_trailing_period(seg):
            end -= 1; changed = True; continue
    return (start, end) if start < end else None


# ---------------------------------------------------------------- span emission
def emit_spans(path, offs, text, scheme, policy, invalid_i="drop"):
    """BIO path over (start,end) items -> spans. An I-X with no open X span is INVALID:
    default 'drop' (the original decoder started a new entity here -- a fragment source)."""
    spans = []
    cur = [None, None, None]  # label,start,end

    def flush():
        if cur[0] is not None:
            t = trim_span(text, cur[1], cur[2], policy)
            if t:
                spans.append((t[0], t[1], cur[0]))
        cur[0] = cur[1] = cur[2] = None

    for tid, off in zip(path, offs):
        tag = scheme.tags[int(tid)]
        ts, te = off
        if tag == "O":
            flush()
        elif tag.startswith("B-"):
            flush()
            cur[0], cur[1], cur[2] = tag[2:], ts, te
        else:
            lab = tag[2:]
            if cur[0] == lab:
                cur[2] = max(cur[2], te)
            elif invalid_i == "start":
                flush()
                cur[0], cur[1], cur[2] = lab, ts, te
            else:
                flush()
    flush()
    return spans


def dedupe_sort(spans):
    return [list(x) for x in sorted(set((int(a), int(b), l) for a, b, l in spans))]


def decode_prepared(prep, text, scheme, policy, bias=None, invalid_i="drop"):
    spans = []
    for S, offs in prep["runs"]:
        if len(offs) == 0:
            continue
        sc = S if bias is None else S + np.asarray(bias)[None, :]
        spans.extend(emit_spans(constrained_viterbi(sc, scheme), offs, text, scheme, policy, invalid_i))
    return dedupe_sort(spans)


# ---------------------------------------------------------------- reference strict scorer
def strict_span_scores(preds, texts, gold, labels):
    """Pooled per-label exact (start,end,label) matching; macro over labels. Dedupes spans.
    Mirrors the keys of the notebook scorer; used for tuning and for synthetic tests."""
    c = {l: [0, 0, 0] for l in labels}
    for d in texts:
        P = set(tuple(x) for x in preds.get(d, []))
        G = set(tuple(x) for x in gold.get(d, []))
        for l in labels:
            pl = {x for x in P if x[2] == l}
            gl = {x for x in G if x[2] == l}
            c[l][0] += len(pl & gl)
            c[l][1] += len(pl - gl)
            c[l][2] += len(gl - pl)
    per = {}
    for l, (tp, fp, fn) in c.items():
        p = tp / (tp + fp) if tp + fp else 0.0
        r = tp / (tp + fn) if tp + fn else 0.0
        f = 2 * p * r / (p + r) if p + r else 0.0
        per[l] = {"tp": tp, "fp": fp, "fn": fn, "precision": p, "recall": r, "f1": f}
    m = len(labels)
    return {"strict_macro_f1": sum(v["f1"] for v in per.values()) / m,
            "macro_precision": sum(v["precision"] for v in per.values()) / m,
            "macro_recall": sum(v["recall"] for v in per.values()) / m,
            "tp": sum(v["tp"] for v in per.values()), "fp": sum(v["fp"] for v in per.values()),
            "fn": sum(v["fn"] for v in per.values()), "documents_evaluated": len(texts), "per_label": per}


def label_f1(preds, gold, doc_ids, label):
    tp = fp = fn = 0
    for d in doc_ids:
        pl = {tuple(x) for x in preds.get(d, []) if x[2] == label}
        gl = {tuple(x) for x in gold.get(d, []) if x[2] == label}
        tp += len(pl & gl); fp += len(pl - gl); fn += len(gl - pl)
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    return 2 * p * r / (p + r) if p + r else 0.0


# ---------------------------------------------------------------- nested evaluation
def train_docs_for(fold_idx, folds):
    return [d for j, f in enumerate(folds) if j != fold_idx for d in f]


def select_structure_nested(cand_preds, folds, gold, labels):
    """For each outer fold: pick the candidate with best pooled F1 on the OTHER folds' docs.
    Returns (chosen_name_per_fold, nested_preds)."""
    chosen, nested = [], {}
    for k, held in enumerate(folds):
        tr = train_docs_for(k, folds)
        best, bf = None, -1.0
        for name, preds in cand_preds.items():
            f = strict_span_scores({d: preds.get(d, []) for d in tr}, {d: None for d in tr},
                                   {d: gold.get(d, []) for d in tr}, labels)["strict_macro_f1"]
            if f > bf + 1e-12:
                best, bf = name, f
        chosen.append(best)
        for d in held:
            nested[d] = cand_preds[best].get(d, [])
    return chosen, nested


def choose_label_bias(f1_by_g, min_gain):
    """f1_by_g: {g: F1 on training docs} with 0.0 present. Shrinks to 0 unless the gain >= min_gain.
    Ties go to the smaller |g|."""
    base = f1_by_g[0.0]
    best_g, best_f = 0.0, base
    for g in sorted(f1_by_g, key=lambda x: (abs(x), x)):
        if f1_by_g[g] > best_f + 1e-12:
            best_g, best_f = g, f1_by_g[g]
    return best_g if best_f >= base + min_gain else 0.0


def tune_biases(grid_preds, base_preds, gold, labels, train_docs, grid, min_gain):
    """grid_preds[(label,g)] = decode of ALL docs with only that label biased by g (g != 0)."""
    out = {}
    for lab in labels:
        f1s = {0.0: label_f1(base_preds, gold, train_docs, lab)}
        for g in grid:
            if g != 0.0:
                f1s[float(g)] = label_f1(grid_preds[(lab, float(g))], gold, train_docs, lab)
        out[lab] = choose_label_bias(f1s, min_gain)
    return out


def gate_decision(nested_fold_f1, base_fold_f1, nested_pooled, base_pooled, min_gain, min_folds, tol):
    stable = sum(c >= b - tol for c, b in zip(nested_fold_f1, base_fold_f1))
    gain = nested_pooled - base_pooled
    return {"pooled_gain": gain, "folds_noninferior": int(stable), "required_folds": min_folds,
            "required_gain": min_gain, "per_fold_delta": [c - b for c, b in zip(nested_fold_f1, base_fold_f1)],
            "passes": bool(gain >= min_gain and stable >= min_folds)}


# ---------------------------------------------------------------- submission checks
def check_prediction_zip(zip_path, expected_ids, texts, labels):
    """Independent invariants: prediction.json at ARCHIVE ROOT, every expected id exactly once,
    spans are [int,int,label] with 0<=start<end<=len(text), label in label set.
    Assumes a {doc_id: [[s,e,label],...]} mapping; otherwise reports the check as skipped."""
    with zipfile.ZipFile(zip_path) as z:
        names = z.namelist()
        if "prediction.json" not in names:
            raise ValueError(f"prediction.json not at archive root; entries={names}")
        raw = z.read("prediction.json").decode("utf-8")
    data = json.loads(raw)
    if not isinstance(data, dict):
        return {"independent_schema_check": "skipped (not a doc_id->spans mapping)", "entries": names}
    exp = [str(x) for x in expected_ids]
    if len(set(exp)) != len(exp):
        raise ValueError("Expected ids collide after str() normalisation.")
    if set(data.keys()) != set(exp):
        raise ValueError(f"Prediction ids differ: missing={sorted(set(exp)-set(data))[:5]} extra={sorted(set(data)-set(exp))[:5]}")
    allowed = set(labels)
    tx = {str(k): v for k, v in texts.items()}
    n = 0
    for d, spans in data.items():
        L = len(tx[d])
        for s in spans:
            if len(s) != 3 or not isinstance(s[0], int) or not isinstance(s[1], int):
                raise ValueError(f"Bad span {s} in {d}")
            if not (0 <= s[0] < s[1] <= L) or s[2] not in allowed:
                raise ValueError(f"Out-of-bounds/unknown-label span {s} in {d}")
            n += 1
    return {"independent_schema_check": "passed", "documents": len(data), "spans": n, "entries": names}


def write_submission_zip(preds, expected_ids, out_dir, zip_name):
    """Local writer used only by the synthetic tests (the kernel uses the notebook's packager)."""
    os.makedirs(out_dir, exist_ok=True)
    mapping = {str(d): [list(s) for s in preds.get(str(d), [])] for d in expected_ids}
    pj = os.path.join(out_dir, "prediction.json")
    with open(pj, "w", encoding="utf-8") as f:
        json.dump(mapping, f, ensure_ascii=False)
    zp = os.path.join(out_dir, zip_name)
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(pj, arcname="prediction.json")
    return pj, zp

print("V1 decoder-repair library loaded.")


## 2. Synthetic unit tests (invented data only)

In [ ]:
# =====================================================================
# SYNTHETIC UNIT TESTS -- every example below is invented; no real data, no model, no GPU.
# Requires the library cell above. Raises if any test fails.
# =====================================================================
import itertools, random, tempfile, traceback

_T_LABELS = ["APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT", "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"]
_T_BIO = ["O"] + [f"{p}-{l}" for l in _T_LABELS for p in "BI"]
_SC = BioScheme(_T_BIO)
_K = _SC.k


def _probs(tags, conf=0.9):
    """Synthetic softmax rows: `conf` mass on the stated tag, remainder spread uniformly."""
    P = np.full((len(tags), _K), (1 - conf) / (_K - 1))
    for i, t in enumerate(tags):
        P[i, _SC.id_of[t]] = conf
    return P


def _toks(text, pieces):
    offs, pos = [], 0
    for p in pieces:
        j = text.index(p, pos)
        offs.append((j, j + len(p)))
        pos = j + len(p)
    return offs


def _decode(text, pieces, tags, mode, policy, conf=0.9, bias=None):
    offs = _toks(text, pieces)
    prep = prepare_document(_probs(tags, conf), offs, text, _SC, mode)
    return decode_prepared(prep, text, _SC, policy, bias)


def _texts_of(text, spans):
    return [(text[s:e], l) for s, e, l in spans]


_results = []


def _run(name, fn):
    try:
        fn()
        _results.append((name, "PASS", ""))
    except Exception as ex:  # noqa
        _results.append((name, "FAIL", "".join(traceback.format_exception_only(type(ex), ex)).strip()[:300]))


def t01_invalid_I_cannot_start_path():
    sc = np.full((3, _K), 1e-4)
    sc[0, _SC.id_of["I-APP"]] = .99
    sc[1, _SC.id_of["B-STAT"]] = .99
    sc[2, _SC.id_of["I-STAT"]] = .99
    sc /= sc.sum(1, keepdims=True)
    path = constrained_viterbi(np.log(sc), _SC)
    tags = [_SC.tags[i] for i in path]
    assert tags[0] != "I-APP", tags
    for a, b in zip(tags, tags[1:]):
        if b.startswith("I-"):
            assert a in (f"B-{b[2:]}", b), (a, b)


def t02_viterbi_matches_bruteforce():
    sc2 = BioScheme(["O", "B-X", "I-X", "B-Y", "I-Y"])
    rng = np.random.default_rng(0)
    for n in range(1, 6):
        for _ in range(40):
            S = rng.normal(size=(n, 5))
            best, bs = None, -1e18
            for seq in itertools.product(range(5), repeat=n):
                tg = [sc2.tags[i] for i in seq]
                if tg[0].startswith("I-"):
                    continue
                if any(b.startswith("I-") and a not in (f"B-{b[2:]}", b) for a, b in zip(tg, tg[1:])):
                    continue
                v = sum(S[t, i] for t, i in enumerate(seq))
                if v > bs:
                    best, bs = seq, v
            got = constrained_viterbi(S, sc2)
            assert abs(sum(S[t, i] for t, i in enumerate(got)) - bs) < 1e-9, (n, got, best)


def t03_emit_drops_invalid_I_by_default():
    offs = [(0, 3), (4, 7), (8, 11)]
    path = [_SC.id_of["I-APP"], _SC.id_of["I-APP"], _SC.o_id]
    assert emit_spans(path, offs, "abc def ghi", _SC, "preserve") == []
    legacy_like = emit_spans(path, offs, "abc def ghi", _SC, "preserve", invalid_i="start")
    assert legacy_like == [(0, 7, "APP")], legacy_like   # the original (buggy) behaviour, kept only for comparison


def t04_multipiece_name_single_span():
    text = "Pardeep Kumar said"
    sp = _decode(text, ["Par", "deep", " Kumar", " said"], ["B-WIT", "I-WIT", "I-WIT", "O"], "word_punct", "preserve")
    assert sp == [[0, 13, "WIT"]], sp


def t05_no_midword_fragments_in_unit_mode():
    text = "Jagdish went home"
    pieces, tags = ["Jag", "dish", " went", " home"], ["B-APP", "O", "O", "O"]
    tok = _decode(text, pieces, tags, "token", "preserve")
    assert tok == [[0, 3, "APP"]], tok                         # token-offset emission reproduces the 'Jag' fragment
    for mode in ("word_punct", "whitespace"):
        units = make_units(text, mode)
        bounds = {s for s, _ in units} | {e for _, e in units}
        for s, e, _ in _decode(text, pieces, tags, mode, "preserve"):
            assert s in bounds and e in bounds, (mode, s, e)    # unit decoding can never cut a word


def t06_whitespace_spanning_entity():
    text = "STATE OF HARYANA ..Respondent"
    pieces = ["STATE", " OF", " HAR", "YANA", " ..", "Respondent"]
    tags = ["B-RESP", "I-RESP", "I-RESP", "I-RESP", "O", "O"]
    assert _decode(text, pieces, tags, "word_punct", "preserve") == [[0, 16, "RESP"]]
    assert _decode(text, pieces, tags, "token", "preserve") == [[0, 16, "RESP"]]


def t07_crpc_final_period():
    text = "under Cr.P.C. and"
    pieces = ["under", " Cr", ".", "P", ".", "C", ".", " and"]
    tags = ["O", "B-STAT", "I-STAT", "I-STAT", "I-STAT", "I-STAT", "I-STAT", "O"]
    assert _texts_of(text, _decode(text, pieces, tags, "word_punct", "preserve")) == [("Cr.P.C.", "STAT")]
    assert _texts_of(text, _decode(text, pieces, tags, "word_punct", "smart")) == [("Cr.P.C.", "STAT")]
    assert _texts_of(text, _decode(text, pieces, tags, "word_punct", "legacy")) == [("Cr.P.C", "STAT")]  # the original bug


def t08_trailing_comma_whitespace_vs_wordpunct():
    text = "Kumar, J. said"
    pieces = ["Kumar", ",", " J", ".", " said"]
    tags = ["B-JUDGE", "O", "O", "O", "O"]
    assert _texts_of(text, _decode(text, pieces, tags, "word_punct", "preserve")) == [("Kumar", "JUDGE")]
    # whitespace units glue the comma to the name: the unit-level B score then needs the ',' piece to look like
    # I-JUDGE, but the model correctly says O, so the whole entity is LOST (recall hit). This is why word_punct is the
    # principled unit and why the probe's whitespace-word decoders are only kept as comparison candidates.
    assert _decode(text, pieces, tags, "whitespace", "preserve") == []


def t09_trim_policy_table():
    cases = [
        ("smart", "see (AIR 1956 SC 316) now", "(AIR 1956 SC 316)", "(AIR 1956 SC 316)"),
        ("smart", "Sec 5) here", "Sec 5)", "Sec 5"),
        ("smart", "(A here", "(A", "A"),
        ("smart", "Union & Ors. x", "Union & Ors.", "Union & Ors."),
        ("smart", "High Court of Punjab. Next", "High Court of Punjab.", "High Court of Punjab"),
        ("smart", "State of U.P. x", "State of U.P.", "State of U.P."),
        ("smart", "R. N. x", "R. N.", "R. N."),
        ("smart", ", Kumar , x", ", Kumar ,", "Kumar"),
        ("soft", "Cr.P.C. x", "Cr.P.C.", "Cr.P.C."),
        ("soft", "Kumar- x", "Kumar-", "Kumar"),
        ("preserve", " Kumar. ", " Kumar. ", "Kumar."),
        ("legacy", "(A.B.) x", "(A.B.)", "A.B"),
    ]
    for pol, text, frag, want in cases:
        s = text.index(frag)
        r = trim_span(text, s, s + len(frag), pol)
        assert r is not None and text[r[0]:r[1]] == want, (pol, frag, r and text[r[0]:r[1]], want)
    assert trim_span("...", 0, 3, "legacy") is None
    assert trim_span("   ", 0, 3, "preserve") is None


def t10_unicode():
    text = "श्री रामेश कुमार और Café Müller Café ok"
    assert [text[s:e] for s, e in make_units(text, "word_punct")][:3] == ["श्री", "रामेश", "कुमार"]
    assert "Café" in [text[s:e] for s, e in make_units(text, "word_punct")]
    pieces = ["श्री", " रामेश", " कुमार", " और", " Café", " Müller", " Café", " ok"]
    tags = ["B-JUDGE", "I-JUDGE", "I-JUDGE", "O", "B-COURT", "I-COURT", "B-WIT", "O"]
    got = _texts_of(text, _decode(text, pieces, tags, "word_punct", "preserve"))
    assert got == [("श्री रामेश कुमार", "JUDGE"), ("Café Müller", "COURT"), ("Café", "WIT")], got


def t11_empty_documents_every_mode():
    for mode in ("token", "whitespace", "word_punct"):
        for text in ("", "   \n ", "abc"):
            prep = prepare_document([], [], text, _SC, mode)
            assert decode_prepared(prep, text, _SC, "smart") == []
            prep = prepare_document(np.zeros((0, _K)), [], text, _SC, mode)
            assert decode_prepared(prep, text, _SC, "preserve") == []
    assert validate_probs([], _K).shape == (0, _K)
    # only special (null / zero-width) tokens
    prep = prepare_document(_probs(["O", "O"]), [None, (0, 0)], "abc", _SC, "word_punct")
    assert decode_prepared(prep, "abc", _SC, "preserve") == []


def _ref_aggregate(windows, k):
    """SYNTHETIC reference aggregator: mean of probabilities for identical (start,end) pieces."""
    acc = {}
    for P, offs in windows:
        for p, o in zip(P, offs):
            if o is not None:
                acc.setdefault(tuple(o), []).append(p)
    keys = sorted(acc)
    return (np.array([np.mean(acc[x], axis=0) for x in keys]) if keys else np.zeros((0, k))), [list(x) for x in keys]


def t12_overlapping_zero_token_and_gap_windows():
    text = "Justice Ram Kumar heard the matter at Delhi High Court today"
    pieces = ["Justice", " Ram", " Kumar", " heard", " the", " matter", " at", " Delhi", " High", " Court", " today"]
    tags = ["O", "B-JUDGE", "I-JUDGE", "O", "O", "O", "O", "B-COURT", "I-COURT", "I-COURT", "O"]
    offs, P = _toks(text, pieces), _probs(tags)
    w1 = (P[:7], offs[:7]); w2 = (P[4:], offs[4:])                     # overlap on pieces 4..6
    w0 = (np.zeros((0, _K)), [])                                       # zero-token window
    agg_p, agg_o = _ref_aggregate([w1, w0, w2], _K)
    assert len(agg_o) == len(pieces)
    prep = prepare_document(agg_p, agg_o, text, _SC, "word_punct")
    assert _texts_of(text, decode_prepared(prep, text, _SC, "smart")) == [("Ram Kumar", "JUDGE"), ("Delhi High Court", "COURT")]
    # window GAP: pieces 3..5 never covered -> an entity may not bridge uncovered non-space text
    gp, go = _ref_aggregate([(P[:3], offs[:3]), (P[6:], offs[6:])], _K)
    prep = prepare_document(gp, go, text, _SC, "token")
    assert len(prep["runs"]) == 2
    tags2 = ["B-JUDGE", "I-JUDGE", "I-JUDGE", "I-JUDGE", "I-JUDGE", "I-JUDGE"]
    P2 = _probs(tags2)
    prep = prepare_document(np.vstack([P2[:3], P2[3:]]), offs[:3] + offs[6:9], text, _SC, "token")
    spans = decode_prepared(prep, text, _SC, "preserve")
    assert all(not (s < offs[3][0] < e) for s, e, _ in spans), spans
    # unsorted + duplicated-window input is re-ordered instead of corrupting spans
    rp, ro = _ref_aggregate([w2, w1], _K)
    assert ro == sorted(ro)


def t13_offset_bounds():
    text = "Ram Kumar"
    offs = [(0, 3), (4, 9), (9, 20), (-3, 2), (5, 5), (7, 4), None, (50, 60)]
    prep = prepare_document(_probs(["B-WIT", "I-WIT"] + ["O"] * 6), offs, text, _SC, "token")
    for mode in ("token", "whitespace", "word_punct"):
        prep = prepare_document(_probs(["B-WIT", "I-WIT"] + ["O"] * 6), offs, text, _SC, mode)
        for s, e, _ in decode_prepared(prep, text, _SC, "preserve"):
            assert 0 <= s < e <= len(text)
    assert clean_offset((4, 99), 9) == (4, 9) and clean_offset((9, 12), 9) is None


def t14_exact_span_metrics():
    L = ["A", "B"]
    texts = {"d1": "x" * 20, "d2": "y" * 20}
    gold = {"d1": [[0, 5, "A"], [6, 9, "B"]], "d2": [[1, 4, "A"]]}
    pred = {"d1": [[0, 5, "A"], [6, 8, "B"]], "d2": [[1, 4, "B"], [1, 4, "B"]]}
    m = strict_span_scores(pred, texts, gold, L)
    a, b = m["per_label"]["A"], m["per_label"]["B"]
    assert (a["tp"], a["fp"], a["fn"]) == (1, 0, 1)             # d2 gold A missed (label differs)
    assert (b["tp"], b["fp"], b["fn"]) == (0, 2, 1)             # (6,8) wrong end; duplicate (1,4,B) counted once
    assert abs(a["f1"] - 2 * 1 * .5 / 1.5) < 1e-12 and b["f1"] == 0.0
    assert abs(m["strict_macro_f1"] - (a["f1"] + b["f1"]) / 2) < 1e-12
    assert strict_span_scores({"d1": []}, {"d1": "x"}, {"d1": []}, L)["strict_macro_f1"] == 0.0


def t15_zip_contents_and_checks():
    texts = {"1": "a" * 30, "2": "b" * 30, "3": ""}
    preds = {"1": [[0, 5, "WIT"]], "2": []}                      # id "3" is empty / missing -> must still appear
    with tempfile.TemporaryDirectory() as td:
        _, zp = write_submission_zip(preds, texts.keys(), td, "s.zip")
        info = check_prediction_zip(zp, texts.keys(), texts, _T_LABELS)
        assert info["independent_schema_check"] == "passed" and info["documents"] == 3 and info["entries"] == ["prediction.json"]
        for bad_preds, why in (({"1": [[0, 99, "WIT"]]}, "bounds"), ({"1": [[0, 5, "NOPE"]]}, "label"), ({"1": [[5, 5, "WIT"]]}, "empty")):
            _, zp2 = write_submission_zip(bad_preds, texts.keys(), td, "b.zip")
            try:
                check_prediction_zip(zp2, texts.keys(), texts, _T_LABELS)
            except ValueError:
                pass
            else:
                raise AssertionError("bad zip accepted: " + why)
        # missing id
        _, zp3 = write_submission_zip(preds, ["1", "2"], td, "m.zip")
        try:
            check_prediction_zip(zp3, texts.keys(), texts, _T_LABELS)
        except ValueError:
            pass
        else:
            raise AssertionError("missing id accepted")
        # nested path (not archive root)
        nz = os.path.join(td, "n.zip")
        with zipfile.ZipFile(nz, "w") as z:
            z.writestr("sub/prediction.json", "{}")
        try:
            check_prediction_zip(nz, texts.keys(), texts, _T_LABELS)
        except ValueError:
            pass
        else:
            raise AssertionError("nested prediction.json accepted")
        # id normalisation collision (1 vs "1")
        try:
            check_prediction_zip(zp, [1, "1"], texts, _T_LABELS)
        except ValueError:
            pass
        else:
            raise AssertionError("id collision accepted")


def t16_bias_moves_precision_recall():
    text = "Ram met Kumar"
    pieces, tags = ["Ram", " met", " Kumar"], ["B-APP", "O", "B-WIT"]
    P = _probs(tags, conf=0.9)
    P[0] = 0.0; P[0, _SC.id_of["B-APP"]] = .45; P[0, _SC.o_id] = .55         # weak APP, O slightly ahead
    prep = prepare_document(P, _toks(text, pieces), text, _SC, "word_punct")
    assert [l for *_, l in decode_prepared(prep, text, _SC, "preserve")] == ["WIT"]
    up = decode_prepared(prep, text, _SC, "preserve", bias_vector(_SC, {"APP": +0.5}))
    assert [l for *_, l in up] == ["APP", "WIT"]
    P2 = P.copy(); P2[0, _SC.id_of["B-APP"]] = .6; P2[0, _SC.o_id] = .4
    prep2 = prepare_document(P2, _toks(text, pieces), text, _SC, "word_punct")
    assert "APP" in [l for *_, l in decode_prepared(prep2, text, _SC, "preserve")]
    dn = decode_prepared(prep2, text, _SC, "preserve", bias_vector(_SC, {"APP": -1.0}))
    assert [l for *_, l in dn] == ["WIT"]


def t17_nested_selection_has_no_leak():
    folds = [["a0", "a1"], ["b0", "b1"], ["c0", "c1"]]
    gold = {d: [[0, 3, "A"]] for f in folds for d in f}
    good = {d: [[0, 3, "A"]] for d in gold}
    bad = {d: [] for d in gold}
    X = {d: (bad[d] if d.startswith("a") else good[d]) for d in gold}   # good everywhere except fold 0
    Y = {d: (good[d] if d.startswith("a") else bad[d]) for d in gold}   # good ONLY on fold 0
    chosen, nested = select_structure_nested({"X": X, "Y": Y}, folds, gold, ["A"])
    assert chosen[0] == "X"                   # fold 0 must not pick Y just because Y is great on fold 0
    assert nested["a0"] == []                 # ... so its held-out predictions come from X
    assert chosen[1] == "Y" or chosen[1] == "X"


def t18_bias_choice_shrinks_to_zero():
    assert choose_label_bias({0.0: .30, -1.0: .3004, -2.0: .25}, min_gain=0.003) == 0.0
    assert choose_label_bias({0.0: .30, -1.0: .34, -2.0: .34}, min_gain=0.003) == -1.0   # tie -> smaller |g|
    assert choose_label_bias({0.0: .30, 0.5: .33, -1.0: .31}, min_gain=0.003) == 0.5


def t19_gate_logic():
    ok = gate_decision([.30, .30, .30, .30, .30], [.29] * 5, .30, .29, .01, 4, .005)
    assert ok["passes"] and ok["folds_noninferior"] == 5
    small = gate_decision([.30] * 5, [.295] * 5, .30, .295, .01, 4, .005)
    assert not small["passes"]                                           # gain 0.005 < 0.010
    one_fold = gate_decision([.25, .33, .33, .33, .33], [.30] * 5, .314, .30, .01, 4, .005)
    assert one_fold["folds_noninferior"] == 4 and one_fold["passes"]
    two_bad = gate_decision([.25, .25, .40, .40, .40], [.30] * 5, .34, .30, .01, 4, .005)
    assert not two_bad["passes"]                                         # big pooled gain but only 3/5 non-inferior


for _n, _f in list(globals().items()):
    if _n.startswith("t") and _n[1:3].isdigit() and callable(_f):
        _run(_n, _f)

print(f"{'test':50s} result")
for _n, _r, _m in _results:
    print(f"{_n:50s} {_r}  {_m}")
_failed = [x for x in _results if x[1] != "PASS"]
print(f"\n{len(_results) - len(_failed)}/{len(_results)} synthetic tests passed (all inputs are synthetic).")
if _failed:
    raise AssertionError(f"{len(_failed)} synthetic decoder tests FAILED -- do not trust the decoder cells.")


## 3. Kernel evaluation, nested gate, conditional single ZIP

In [ ]:
# =====================================================================
# KERNEL CELL -- preflight + OOF decoder evaluation + nested gate + (conditional) ONE candidate ZIP
# Run in the completed V1 five-fold kernel, AFTER the cell that defines `oof_window_records_by_doc` and
# `ensemble_test_window_records_by_doc` (cell 99), BEFORE final package selection.
# Needs the library cell above. NO model training. Never overwrites earlier submissions. Never auto-submits.
# =====================================================================
import time, collections

_P = lambda name, default: globals().get(name, default)

# ---- predeclared gate / search settings (decided BEFORE seeing any decoder score; do not tune after the fact) ----
GATE_MIN_GAIN = _P("GATE_MIN_GAIN", 0.010)       # nested pooled Strict Macro-F1 gain over the current aggregate decoder
GATE_MIN_FOLDS = _P("GATE_MIN_FOLDS", 4)         # nested fold F1 must be within tolerance of baseline on >= 4/5 folds
GATE_FOLD_TOL = _P("GATE_FOLD_TOL", 0.005)
BIAS_GRID = _P("BIAS_GRID", (-3.0, -2.0, -1.0, -0.5, 0.0, 0.5))   # additive log-score bias per label (0 = off)
BIAS_MIN_LABEL_GAIN = _P("BIAS_MIN_LABEL_GAIN", 0.003)            # a label bias is kept only if train-fold F1 gain >= this
STAGE2_STRUCTURE = _P("STAGE2_STRUCTURE", "wp_viterbi_preserve")  # predeclared (word/punct units, model decides punctuation)
ALLOW_BASELINE_MISMATCH = _P("ALLOW_BASELINE_MISMATCH", False)
# supplied OOF metrics for the aggregate decoder (oof_validation_metrics.json) -> kernel-state reproduction check
EXPECTED_AGG = {"strict_macro_f1": 0.26480271544599987, "tp": 3703, "fp": 7702, "fn": 4690}

_STRUCTURES = {                         # name -> (unit mode, boundary policy)
    "token_viterbi_preserve": ("token", "preserve"),
    "ws_viterbi_preserve": ("whitespace", "preserve"),
    "ws_viterbi_soft": ("whitespace", "soft"),
    "ws_viterbi_legacy": ("whitespace", "legacy"),
    "wp_viterbi_preserve": ("word_punct", "preserve"),
    "wp_viterbi_soft": ("word_punct", "soft"),
    "wp_viterbi_smart": ("word_punct", "smart"),
}

# ---------------------------------------------------------------- preflight
_REQUIRED = ["oof_window_records_by_doc", "ensemble_test_window_records_by_doc", "OOF_TEXTS", "OOF_GOLD",
             "CV_TEST_TEXTS", "CV_OUTPUT_DIR", "CV_DOCUMENT_FOLDS", "LABELS", "BIO_LABELS",
             "aggregate_window_probabilities", "predict_phase2_from_window_records", "score_phase2_predictions",
             "validate_cv_prediction_mapping", "package_codabench_submission", "validate_submission_zip",
             "BioScheme", "prepare_document", "decode_prepared", "bias_vector"]
_missing = [n for n in _REQUIRED if n not in globals()]
if _missing:
    raise RuntimeError("Run the library cell and the completed V1 5-fold kernel first. Missing: " + ", ".join(_missing))

_PROBE_DIR = os.path.join(os.path.dirname(str(CV_OUTPUT_DIR).rstrip("/")), "V1_SpanDecoder_Repair_Reviewed")
os.makedirs(_PROBE_DIR, exist_ok=True)

_scheme = BioScheme(BIO_LABELS)
if set(_scheme.labels) != set(LABELS):
    raise RuntimeError(f"LABELS {sorted(LABELS)} != BIO_LABELS-derived {sorted(_scheme.labels)}")
_labels = list(LABELS)

_T = {str(k): v for k, v in OOF_TEXTS.items()}
if len(_T) != len(OOF_TEXTS):
    raise RuntimeError("OOF document ids collide after str() normalisation.")
_G = {}
for k, v in OOF_GOLD.items():
    spans = [list(x) for x in v]
    for x in spans:
        if len(x) != 3 or not isinstance(x[2], str):
            raise RuntimeError(f"Unexpected gold span format {x!r}; expected [start, end, label].")
    _G[str(k)] = [[int(a), int(b), l] for a, b, l in spans]
_folds = [[str(d) for d in f] for f in CV_DOCUMENT_FOLDS]
_all_docs = [d for f in _folds for d in f]
if len(_all_docs) != len(set(_all_docs)) or set(_all_docs) != set(_T):
    raise RuntimeError("CV_DOCUMENT_FOLDS must partition the OOF documents exactly once each.")
_TT = {str(k): v for k, v in CV_TEST_TEXTS.items()}
if len(_TT) != len(CV_TEST_TEXTS):
    raise RuntimeError("Test document ids collide after str() normalisation.")
print(f"Preflight OK: {len(_all_docs)} OOF docs in {len(_folds)} folds, {len(_TT)} test docs, {len(_labels)} labels.")


def _aggregate(records_by_doc, doc_id):
    recs = records_by_doc.get(doc_id, [])
    if recs is None or len(recs) == 0:
        return np.zeros((0, _scheme.k)), []
    try:
        p, offs = aggregate_window_probabilities(recs, doc_id)
    except Exception as ex:
        raise RuntimeError(f"aggregate_window_probabilities failed for doc {doc_id}: {ex!r}") from ex
    return p, list(offs)


def _nb_score(preds, doc_ids):
    ids = list(doc_ids)
    return score_phase2_predictions({d: preds.get(d, []) for d in ids}, {d: _T[d] for d in ids},
                                    {d: _G.get(d, []) for d in ids})


# ---------------------------------------------------------------- baselines + reproduction check
_t0 = time.time()
_AGG_OOF = {d: _aggregate(oof_window_records_by_doc, d) for d in _all_docs}
_base_window = {str(k): v for k, v in predict_phase2_from_window_records(oof_window_records_by_doc, OOF_TEXTS, mode="window").items()}
_base_agg = {str(k): v for k, v in predict_phase2_from_window_records(oof_window_records_by_doc, OOF_TEXTS, mode="aggregate").items()}
_m_agg = _nb_score(_base_agg, _all_docs)
_repro = {k: (float(_m_agg[k]) if k == "strict_macro_f1" else int(_m_agg[k])) for k in EXPECTED_AGG}
_repro_ok = (abs(_repro["strict_macro_f1"] - EXPECTED_AGG["strict_macro_f1"]) < 1e-4
             and all(_repro[k] == EXPECTED_AGG[k] for k in ("tp", "fp", "fn")))
print("Baseline aggregate re-scored:", _repro, "| matches supplied metrics JSON:", _repro_ok)
if not _repro_ok and not ALLOW_BASELINE_MISMATCH:
    raise RuntimeError("Kernel state does not reproduce the supplied OOF baseline; refusing to continue. "
                       "(Set ALLOW_BASELINE_MISMATCH=True only if you know why.)")
_ref = strict_span_scores(_base_agg, {d: 1 for d in _all_docs}, _G, _labels)
_scorer_agrees = (abs(_ref["strict_macro_f1"] - float(_m_agg["strict_macro_f1"])) < 1e-9
                  and (_ref["tp"], _ref["fp"], _ref["fn"]) == (int(_m_agg["tp"]), int(_m_agg["fp"]), int(_m_agg["fn"])))
print("Reference exact-span scorer agrees with notebook scorer:", _scorer_agrees)

# ---------------------------------------------------------------- prepare + alignment diagnostics
_PREP = {}
for _mode in sorted({m for m, _ in _STRUCTURES.values()}):
    _PREP[_mode] = {d: prepare_document(*_AGG_OOF[d], _T[d], _scheme, _mode) for d in _all_docs}
print(f"Prepared OOF docs in {time.time() - _t0:.0f}s")

_diag = {}
for _mode in ("whitespace", "word_punct"):
    agg = collections.Counter()
    for d in _all_docs:
        agg.update(_PREP[_mode][d]["diag"])
    _diag[_mode] = {"tokens": agg["tokens"], "units": agg["units"],
                    "empty_unit_fraction": agg["empty_units"] / max(1, agg["units"]),
                    "tokens_straddling_units_fraction": agg["straddling_tokens"] / max(1, agg["tokens"] - agg["null_offsets"]),
                    "unassigned_token_fraction": agg["unassigned_tokens"] / max(1, agg["tokens"] - agg["null_offsets"])}
# gold-boundary alignment against each unit type (how many gold spans CAN a unit decoder reproduce at all?)
for _mode in ("whitespace", "word_punct"):
    ok = tot = 0
    for d in _all_docs:
        units = make_units(_T[d], _mode)
        st = {s for s, _ in units}; en = {e for _, e in units}
        for s, e, _l in _G.get(d, []):
            tot += 1
            ok += (s in st and e in en)
    _diag[_mode]["gold_spans_aligned_to_unit_boundaries"] = ok / max(1, tot)
print("Alignment diagnostics (real OOF data):")
for m, v in _diag.items():
    print(f"  {m:11s}", {k: (round(x, 4) if isinstance(x, float) else x) for k, x in v.items()})

# ---------------------------------------------------------------- stage 1: decoder structures
_cand = {}
_metrics_all, _metrics_fold = {}, {}


def _decode_all(mode, policy, bias=None, docs=None):
    docs = _all_docs if docs is None else docs
    return {d: decode_prepared(_PREP[mode][d], _T[d], _scheme, policy, bias) for d in docs}


_metrics_all["baseline_window"] = _nb_score(_base_window, _all_docs)
_metrics_all["baseline_overlap_aggregate"] = _m_agg
_metrics_fold["baseline_overlap_aggregate"] = [_nb_score(_base_agg, f) for f in _folds]
for name, (mode, policy) in _STRUCTURES.items():
    t1 = time.time()
    _cand[name] = _decode_all(mode, policy)
    _metrics_all[name] = _nb_score(_cand[name], _all_docs)
    _metrics_fold[name] = [_nb_score(_cand[name], f) for f in _folds]
    print(f"  decoded {name:26s} in {time.time() - t1:5.0f}s")


def _row(name, m):
    return (f"{name:30s} F1={m['strict_macro_f1']:.4f} P={m['macro_precision']:.4f} "
            f"R={m['macro_recall']:.4f} TP={m['tp']} FP={m['fp']} FN={m['fn']}")


print("\nPOOLED OOF (all docs; in-sample over decoder choice -- optimistic, nested numbers below are the ones to trust):")
for name, m in _metrics_all.items():
    print("  " + _row(name, m))

_base_fold_f1 = [float(m["strict_macro_f1"]) for m in _metrics_fold["baseline_overlap_aggregate"]]
_base_pooled = float(_m_agg["strict_macro_f1"])

_chosen1, _nested1 = select_structure_nested(_cand, _folds, _G, _labels)
_n1_pooled = _nb_score(_nested1, _all_docs)
_n1_fold = [float(_nb_score(_nested1, f)["strict_macro_f1"]) for f in _folds]
_final1 = max(_cand, key=lambda n: float(_metrics_all[n]["strict_macro_f1"]))
_modal, _modal_n = collections.Counter(_chosen1).most_common(1)[0]
_stable1 = (_final1 == _modal and _modal_n >= 3)
_gate1 = gate_decision(_n1_fold, _base_fold_f1, float(_n1_pooled["strict_macro_f1"]), _base_pooled,
                       GATE_MIN_GAIN, GATE_MIN_FOLDS, GATE_FOLD_TOL)
_gate1["structure_selection_stable"] = bool(_stable1)
_gate1["passes"] = bool(_gate1["passes"] and _stable1)
print("\nSTAGE 1 nested structure selection -- fold-wise choice:", _chosen1, "| refit-on-all choice:", _final1)
print("  " + _row("nested stage-1 (held-out)", _n1_pooled))
print("  gate:", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in _gate1.items() if k != "per_fold_delta"},
      "per-fold delta:", [round(x, 4) for x in _gate1["per_fold_delta"]])

# ---------------------------------------------------------------- stage 2: nested per-label bias tuning
_s2_mode, _s2_policy = _STRUCTURES[STAGE2_STRUCTURE]
_grid = tuple(float(g) for g in BIAS_GRID)
if 0.0 not in _grid:
    _grid = (0.0,) + _grid
_base_s2 = _cand[STAGE2_STRUCTURE]
_grid_preds = {}
_t2 = time.time()
for lab in _labels:
    for g in _grid:
        if g != 0.0:
            _grid_preds[(lab, g)] = _decode_all(_s2_mode, _s2_policy, bias_vector(_scheme, {lab: g}))
    print(f"  bias grid decoded for {lab:10s} ({time.time() - _t2:.0f}s elapsed)")
_nested2, _fold_bias = {}, []
for k, held in enumerate(_folds):
    beta = tune_biases(_grid_preds, _base_s2, _G, _labels, train_docs_for(k, _folds), _grid, BIAS_MIN_LABEL_GAIN)
    _fold_bias.append(beta)
    vec = bias_vector(_scheme, beta)
    for d in held:
        _nested2[d] = decode_prepared(_PREP[_s2_mode][d], _T[d], _scheme, _s2_policy, vec)
_n2_pooled = _nb_score(_nested2, _all_docs)
_n2_fold = [float(_nb_score(_nested2, f)["strict_macro_f1"]) for f in _folds]
_gate2 = gate_decision(_n2_fold, _base_fold_f1, float(_n2_pooled["strict_macro_f1"]), _base_pooled,
                       GATE_MIN_GAIN, GATE_MIN_FOLDS, GATE_FOLD_TOL)
_final_bias = tune_biases(_grid_preds, _base_s2, _G, _labels, _all_docs, _grid, BIAS_MIN_LABEL_GAIN)
_insample2 = _nb_score(_decode_all(_s2_mode, _s2_policy, bias_vector(_scheme, _final_bias)), _all_docs)
print(f"\nSTAGE 2 nested bias tuning on {STAGE2_STRUCTURE}")
print("  per-fold chosen biases:", _fold_bias)
print("  " + _row("nested stage-2 (held-out)", _n2_pooled))
print("  gate:", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in _gate2.items() if k != "per_fold_delta"},
      "per-fold delta:", [round(x, 4) for x in _gate2["per_fold_delta"]])
print("  refit-on-all biases:", _final_bias, "(in-sample F1 %.4f -- optimistic, do not quote)" % _insample2["strict_macro_f1"])

print("\nPer-label F1: baseline aggregate vs nested held-out stage 1 / stage 2")
for lab in _labels:
    print(f"  {lab:10s} base={_m_agg['per_label'][lab]['f1']:.4f}  s1={_n1_pooled['per_label'][lab]['f1']:.4f}  "
          f"s2={_n2_pooled['per_label'][lab]['f1']:.4f}   (s2 P={_n2_pooled['per_label'][lab]['precision']:.3f} "
          f"R={_n2_pooled['per_label'][lab]['recall']:.3f})")

# ---------------------------------------------------------------- choose + (maybe) one ZIP
_options = []
if _gate1["passes"]:
    _options.append(("stage1", float(_n1_pooled["strict_macro_f1"])))
if _gate2["passes"]:
    _options.append(("stage2", float(_n2_pooled["strict_macro_f1"])))
_choice = max(_options, key=lambda x: x[1])[0] if _options else None

_report = {
    "experiment": "V1_SpanDecoder_Repair_Reviewed", "training_performed": False,
    "baseline_reproduction": {"rescored": _repro, "expected": EXPECTED_AGG, "matches": bool(_repro_ok),
                              "reference_scorer_agrees_with_notebook_scorer": bool(_scorer_agrees)},
    "alignment_diagnostics": _diag,
    "settings": {"GATE_MIN_GAIN": GATE_MIN_GAIN, "GATE_MIN_FOLDS": GATE_MIN_FOLDS, "GATE_FOLD_TOL": GATE_FOLD_TOL,
                 "BIAS_GRID": list(_grid), "BIAS_MIN_LABEL_GAIN": BIAS_MIN_LABEL_GAIN, "STAGE2_STRUCTURE": STAGE2_STRUCTURE},
    "pooled_oof_in_sample": {n: {k: (float(m[k]) if k in ("strict_macro_f1", "macro_precision", "macro_recall") else int(m[k]))
                                for k in ("strict_macro_f1", "macro_precision", "macro_recall", "tp", "fp", "fn")}
                             for n, m in _metrics_all.items()},
    "baseline_fold_f1": _base_fold_f1,
    "stage1": {"fold_choice": _chosen1, "refit_choice": _final1, "nested_fold_f1": _n1_fold,
               "nested_pooled": {k: float(_n1_pooled[k]) for k in ("strict_macro_f1", "macro_precision", "macro_recall")} | {k: int(_n1_pooled[k]) for k in ("tp", "fp", "fn")},
               "gate": _gate1},
    "stage2": {"structure": STAGE2_STRUCTURE, "fold_biases": _fold_bias, "refit_biases": _final_bias, "nested_fold_f1": _n2_fold,
               "nested_pooled": {k: float(_n2_pooled[k]) for k in ("strict_macro_f1", "macro_precision", "macro_recall")} | {k: int(_n2_pooled[k]) for k in ("tp", "fp", "fn")},
               "nested_per_label": {l: _n2_pooled["per_label"][l] for l in _labels}, "gate": _gate2},
    "candidate_choice": _choice,
    "caveat": ("Nested OOF removes decoder-choice leakage but not the uncertainty of 70 documents / 5 folds; fold-to-fold "
               "F1 SD is ~0.03, larger than the 0.010 gate. OOF gains need not transfer to the public test set."),
}

if _choice is None:
    print("\nNO ZIP CREATED: neither the nested stage-1 nor the nested stage-2 decoder passed the predeclared gate "
          f"(>= +{GATE_MIN_GAIN:.3f} pooled held-out F1 and non-inferior within {GATE_FOLD_TOL:.3f} on >= {GATE_MIN_FOLDS}/5 folds).")
else:
    if _choice == "stage1":
        _mode, _policy = _STRUCTURES[_final1]
        _vec = None
    else:
        _mode, _policy = _s2_mode, _s2_policy
        _vec = bias_vector(_scheme, _final_bias)
    _AGG_TEST = {d: _aggregate(ensemble_test_window_records_by_doc, d) for d in _TT}
    _test_pred = {d: decode_prepared(prepare_document(*_AGG_TEST[d], _TT[d], _scheme, _mode), _TT[d], _scheme, _policy, _vec)
                  for d in _TT}
    validate_cv_prediction_mapping(_test_pred, CV_TEST_TEXTS, LABELS)
    _pred_path, _zip_path = package_codabench_submission(
        predictions=_test_pred, output_dir=_PROBE_DIR, zip_filename="submission_v1_decoder_repair_nested_gated.zip")
    validate_submission_zip(_zip_path, expected_doc_ids=CV_TEST_TEXTS.keys())
    _chk = check_prediction_zip(_zip_path, list(_TT), _TT, _labels)
    _report["submission_zip"] = _zip_path
    _report["independent_zip_check"] = _chk
    _report["test_decoder"] = {"choice": _choice, "mode": _mode, "policy": _policy,
                               "biases": (_final_bias if _choice == "stage2" else None)}
    print(f"\nONE candidate ZIP created ({_choice}): {_zip_path}\n  independent check: {_chk}")
    print("  Submit it only as a deliberate leaderboard experiment; no public score is implied.")

with open(os.path.join(_PROBE_DIR, "decoder_repair_report.json"), "w", encoding="utf-8") as f:
    json.dump(_report, f, indent=2, ensure_ascii=False, default=float)
print("Report:", os.path.join(_PROBE_DIR, "decoder_repair_report.json"), f"| total {time.time() - _t0:.0f}s")
